# Assertion Calibration — TPR / TNR / FPR / FNR

Compares auto-computed assertion scores (`results.jsonl`) against human labels (`labels.json`) to measure how well each binary assertion tracks human judgment.

Loads directly from `../` so no file copying needed. Run the annotation server and label some traces first.

**Definitions** (auto = assertion scorer, human = annotator):
| | Human True | Human False |
|---|---|---|
| **Auto True** | TP | FP |
| **Auto False** | FN | TN |

- **TPR** (recall): of real positives, how many did the scorer catch? TP / (TP + FN)
- **TNR** (specificity): of real negatives, how many did it correctly pass? TN / (TN + FP)
- **FPR**: false alarm rate = 1 − TNR
- **FNR**: miss rate = 1 − TPR

> **Note:** metrics are only computed on traces where the human has set a non-null assertion value (True or False — N/A skipped). With few labeled traces numbers will be noisy; treat them as directional.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

EVAL_DIR = Path("..").resolve()   # evals/intelligence_agent/

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:.3f}".format)

# Sanity check — catch wrong working directory early
for fname in ["results.jsonl", "labels.json"]:
    p = EVAL_DIR / fname
    assert p.exists(), f"Expected {p} — is the kernel cwd the notebook directory?"
print(f"EVAL_DIR: {EVAL_DIR}  ✓")

## Load results and labels

In [ ]:
def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]

# Auto-computed assertion scores
results_raw = load_jsonl(EVAL_DIR / "results.jsonl")
results_df = pd.json_normalize(results_raw, sep=".")
results_df["key"] = results_df["run_id"] + ":" + results_df["id"].astype(str)
assertion_cols = [c for c in results_df.columns if c.startswith("assertions.")]

print(f"Results  : {len(results_df)} rows across {results_df['run_id'].nunique()} runs, "
      f"{results_df['id'].nunique()} unique tuples, {len(assertion_cols)} assertions")

In [ ]:
# Human annotation labels
labels_path = EVAL_DIR / "labels.json"
if not labels_path.exists():
    raise FileNotFoundError(f"No labels file found at {labels_path}. Label some traces first.")

labels_raw = json.loads(labels_path.read_text())

# Flatten to one row per (trace_key, assertion_key, human_value)
# Exclude null/N/A entries — those are not meaningful signal
label_rows = []
for key, entry in labels_raw.items():
    for akey, val in (entry.get("human_assertions") or {}).items():
        if val is not None:
            label_rows.append({"key": key, "assertion": akey, "human": bool(val)})

labels_df = pd.DataFrame(label_rows)
print(f"Labels   : {len(labels_raw)} annotated traces, "
      f"{len(labels_df)} non-null assertion ratings across "
      f"{labels_df['assertion'].nunique()} assertion keys")

## Align auto scores with human labels

In [ ]:
# Melt auto assertions to long form: one row per (trace_key, assertion_key, auto_value)
auto_long = (
    results_df[["key"] + assertion_cols]
    .melt(id_vars="key", var_name="assertion", value_name="auto")
)
auto_long["assertion"] = auto_long["assertion"].str.removeprefix("assertions.")

# Inner join: only keep (key, assertion) pairs that have BOTH a human label and an auto score
merged = labels_df.merge(auto_long, on=["key", "assertion"], how="inner")
merged = merged.dropna(subset=["auto"])   # drop traces where auto score is null (e.g. errors)
merged["auto"] = merged["auto"].astype(bool)

print(f"Matched {len(merged)} (trace × assertion) pairs for evaluation")
print(f"Assertions covered: {sorted(merged['assertion'].unique())}")
merged.head(10)

## Confusion matrix metrics per assertion

In [ ]:
def confusion_metrics(group):
    TP = ((group["auto"] == True)  & (group["human"] == True)).sum()
    TN = ((group["auto"] == False) & (group["human"] == False)).sum()
    FP = ((group["auto"] == True)  & (group["human"] == False)).sum()
    FN = ((group["auto"] == False) & (group["human"] == True)).sum()
    TPR = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    TNR = TN / (TN + FP) if (TN + FP) > 0 else np.nan
    return pd.Series({
        "n": len(group),
        "TP": int(TP), "TN": int(TN), "FP": int(FP), "FN": int(FN),
        "TPR": TPR, "TNR": TNR,
        "FPR": 1 - TNR if not np.isnan(TNR) else np.nan,
        "FNR": 1 - TPR if not np.isnan(TPR) else np.nan,
    })

metrics = (
    merged.groupby("assertion")
    .apply(confusion_metrics, include_groups=False)
    .reset_index()
    .sort_values("n", ascending=False)
)
metrics[["n","TP","TN","FP","FN"]] = metrics[["n","TP","TN","FP","FN"]].astype(int)

print("NaN = not enough signal (e.g. no positive cases in sample for TPR)\n")
metrics

## Visualise — TPR vs TNR per assertion

Top-right = good. Low TPR = missing real positives (under-flagging). Low TNR = too many false alarms. Points sized by sample count.

In [ ]:
plot_df = metrics.dropna(subset=["TPR", "TNR"])

fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    plot_df["TNR"], plot_df["TPR"],
    s=plot_df["n"] * 40,
    alpha=0.75, edgecolors="white", linewidths=0.8
)
for _, row in plot_df.iterrows():
    ax.annotate(row["assertion"], (row["TNR"], row["TPR"]),
                fontsize=8, ha="left", va="bottom",
                xytext=(4, 4), textcoords="offset points")

ax.axhline(0.8, color="gray", linestyle="--", linewidth=0.8)
ax.axvline(0.8, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("TNR (specificity) — low = too many false alarms")
ax.set_ylabel("TPR (recall) — low = missing real positives")
ax.set_title("Assertion calibration (bubble size = sample count)")
ax.set_xlim(-0.05, 1.1)
ax.set_ylim(-0.05, 1.1)
plt.tight_layout()
plt.show()

if len(plot_df) < len(metrics):
    skipped = metrics[metrics["TPR"].isna() | metrics["TNR"].isna()]["assertion"].tolist()
    print(f"Skipped from plot (insufficient sample): {skipped}")

## Disagreements — where auto and human diverge

In [ ]:
disagreements = merged[merged["auto"] != merged["human"]].copy()
disagreements["error_type"] = np.where(
    disagreements["auto"] & ~disagreements["human"], "FP", "FN"
)

print(f"{len(disagreements)} disagreements out of {len(merged)} rated pairs "
      f"({100*len(disagreements)/len(merged):.0f}%)\n")
disagreements[["key", "assertion", "auto", "human", "error_type"]]